# Phase 4 — Évaluation finale

Évalue les 4 conditions (Baseline, DPO, RLHF, RAG) sur ETHICS.

## ⚠️ Datasets à attacher

- `adl-dpo-adapter` (notebook 01)
- `adl-rlhf-model` (notebook 03)
- `adl-reward-model` est facultatif ici (l'eval n'en a pas besoin, le RM est déjà fusionné dans le RLHF)

**Durée** : ~30 min pour 4 conditions × 500 exemples.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# Optionnel : pour la génération synthétique (data/generate_synthetic.py)
# from kaggle_secrets import UserSecretsClient
# os.environ["ANTHROPIC_API_KEY"] = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")

In [ ]:
!pip install -q -U bitsandbytes transformers==4.46.3 trl==0.12.0 peft==0.14.0 \
    accelerate==1.2.0 datasets==3.2.0 sentence-transformers faiss-cpu \
    anthropic openai pyarrow==17.0.0 tqdm

In [ ]:
!rm -rf /kaggle/working/adl
!git clone https://github.com/FeelTheFloww/adl-ethics.git /kaggle/working/adl
%cd /kaggle/working/adl

In [ ]:
# Adapte selon tes datasets
DPO_ZIP  = "/kaggle/input/adl-dpo-adapter/dpo_model.zip"
RLHF_ZIP = "/kaggle/input/adl-rlhf-model/rlhf_model.zip"

import os, zipfile
os.makedirs("results/dpo_model",  exist_ok=True)
os.makedirs("results/rlhf_model", exist_ok=True)
with zipfile.ZipFile(DPO_ZIP)  as z: z.extractall("results/dpo_model")
with zipfile.ZipFile(RLHF_ZIP) as z: z.extractall("results/rlhf_model")
print("DPO:",  os.listdir("results/dpo_model"))
print("RLHF:", os.listdir("results/rlhf_model"))

In [ ]:
!python eval/evaluate_ethics.py \
    --base_model     Qwen/Qwen2.5-1.5B-Instruct \
    --dpo_adapter    results/dpo_model \
    --rlhf_adapter   results/rlhf_model \
    --ethical_corpus data/ethical_corpus_v2.json \
    --output_path    results/eval_results.json \
    --n_per_cat      100

## Figure KL (optionnel)

In [ ]:
# Si tu as gardé le checkpoint PPO dans le dataset RLHF
!python eval/plot_ppo_kl.py \
    --trainer_state results/rlhf_model/checkpoint-125/trainer_state.json \
    --out results/ppo_kl_divergence.png || echo "(checkpoint trainer_state.json non trouvé, skip)"